# Linear Regression: PyTorch 

In [10]:
import torch

## Data

Three clean points that follow `y = 2x`. The model never sees that rule — it only sees inputs and answers, and has to recover the rule on its own.

| x | y |
|---|---|
| 1 | 2 |
| 2 | 4 |
| 3 | 6 |

> Note: `Variable(...)` was old PyTorch syntax (pre-2018). Plain tensors do everything `Variable` used to do.

In [11]:
x_data = torch.tensor([[1.0], [2.0], [3.0]])
y_data = torch.tensor([[2.0], [4.0], [6.0]])

## Model

Defining a model in PyTorch has two steps:
1. **`__init__`** : declare the layers and parameters
2.  **`forward`**  : describe how data flows through the model

`nn.Linear(1, 1)` creates a linear layer with one input feature and one output. Internally it holds a weight `w` and a bias `b`, exactly like the `w` and `b` we declared manually before — we just don't have to manage them ourselves anymore.

In [12]:
class LinearRegressionModel(torch.nn.Module):

    def __init__(self):
        super(LinearRegressionModel, self).__init__()
        self.linear = torch.nn.Linear(1, 1)

    def forward(self, x):
        return self.linear(x)

## Loss and Optimizer

Every manual step from `linear_regression_manual.ipynb` now has a one-line equivalent:

| By hand | With PyTorch tools |
|---|---|
| `w`, `b` as tensors with `requires_grad=True` | `nn.Linear(1, 1)` holds them |
| Hand-written MSE formula | `nn.MSELoss()` |
| `w.data = w.data - lr * w.grad.data` | `optimizer.step()` |
| `w.grad.data.zero_()` | `optimizer.zero_grad()` |

In [13]:
our_model = LinearRegressionModel()
criterion = torch.nn.MSELoss(reduction='sum')
optimizer = torch.optim.SGD(our_model.parameters(), lr=0.01)

## Training Loop

Each epoch does the same four steps as before — forward pass, compute loss, zero gradients, backward pass, update weights — just with the library handling the parameter updates.

In [14]:
for epoch in range(500):
    pred_y = our_model(x_data)
    loss = criterion(pred_y, y_data)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if epoch % 50 == 0:
        print(f'epoch {epoch}, loss {loss.item()}')

epoch 0, loss 73.02423858642578
epoch 50, loss 0.3401259481906891
epoch 100, loss 0.16493037343025208
epoch 150, loss 0.07997623831033707
epoch 200, loss 0.038781266659498215
epoch 250, loss 0.018805447965860367
epoch 300, loss 0.009118922054767609
epoch 350, loss 0.0044218916445970535
epoch 400, loss 0.0021441837307065725
epoch 450, loss 0.0010397512232884765


## Results

After training, we can read the learned weight and bias directly from the layer, and test it on a new input it never saw during training.

In [15]:
w, b = our_model.linear.weight.item(), our_model.linear.bias.item()
print(f"learned: y = {w:.3f}x + {b:.3f}")

new_x = torch.tensor([[4.0]])
pred_y = our_model(new_x)
print(f"predict x=4 → {pred_y.item():.4f}  (true answer: 8.0)")

learned: y = 1.985x + 0.034
predict x=4 → 7.9740  (true answer: 8.0)
